#<font color="Green">**Notebook Purpose**</font>

This notebook constructs a per-cluster breakdown of the demographic, clinical,
and comorbidity variables shown in manuscript Table 1, in response to **Referee
4's third comment** (*"It would be interesting to see Table 1 broken down by
cluster, though this would need to be presented graphically."*).

A full Table 1 replicated 30 times would be unwieldy. Instead, this notebook
produces **two candidate deliverables** so the more readable one can be selected
for the supplement:

- **Experiment A — Long-form eTable.** 30 rows (one per primary cluster, grouped
  by therapy category in the same order as manuscript Tables 2/3 and eTable 2)
  with one column per Table 1 variable. Categorical variables (race, region)
  are shown as % per cluster; age, HbA1c, and BMI are summarized as mean (SD)
  per cluster for compactness, as the categorical breakdowns from Table 1 would
  expand to 15+ columns at the cluster level.
- **Experiment B — Standardized heatmap.** Same 30 clusters as rows, the key
  characteristics as columns, with cell color encoding deviation from the
  cohort mean (z-score across clusters) and the raw value annotated. This is
  the explicitly *graphical* option the reviewer mentioned.

Both experiments use the same baseline-value definitions as the revised Table 1:
HbA1c and BMI are the closest measurement in the 183-day pre-GLM window, and
comorbidities are documented HF/CKD before June 1, 2019.

**Rationale:** Manuscript Table 1 already summarizes these characteristics for
the full primary cohort, and eTable 2 already gives top prescriptions and HF/CKD
prevalence per cluster, but neither shows demographic, clinical, and comorbidity
characteristics jointly at the cluster level. The reviewer's hypothesis — that
the disparities described in the Discussion (Hispanic patients under-represented
in early GLP-1 RA clusters, older adults in monotherapy clusters, etc.) would be
easier to interpret with a side-by-side view of cluster characteristics — is the
goal of these visualizations.

---

###<font color="Red"> Required Data </font>

To run the code blocks in this notebook, you will need the following **cleaned**
CSVs (output from `CohortDatasetCreation.ipynb`):

1. **`patient_demographics.csv`** — columns: `patient_id`, `sex`, `race/ethnicity`, `year_of_birth`, `month_year_death`, `patient_regional_location`
2. **`patient_comorbidities.csv`** — columns: `patient_id`, `date`, `HF`, `CKD`

Plus the baseline-value outputs from `get_closest_continuous_value_after_t0`
(already generated for the revised Table 1):

3. **`baseline_a1c.csv`** — columns: `patient_id`, `baseline_a1c_value`, `baseline_a1c_days_from_t0`
4. **`baseline_bmi.csv`** — columns: `patient_id`, `baseline_bmi_value`, `baseline_bmi_days_from_t0`

Plus the cluster-assignment dictionaries exported from `ClusteringAnalysis.ipynb`
(each a `{cluster_id: [patient_ids]}` dict):

5. **`monotherapy_patients.pkl`**
6. **`dual_therapy_patients.pkl`**
7. **`complex_therapy_patients.pkl`**
8. **`variant_therapy_patients.pkl`**
9. **`GLP_1_therapy_patients.pkl`**

And the manuscript .docx to pull canonical cluster names from Table 2:

10. **`DOM_T2D_Manuscript_v2026-04-16.docx`** — source of cluster names (Table 2, python-docx index 1)

Note: the early-disengagement clusters (`early_dropout_patients.pkl`) are not
loaded here. Table 1 is reported for the primary analytic cohort (N = 9,327),
which excludes those clusters; this breakdown mirrors that scope.

## Section 1 — Imports & Data Loading

In [ ]:
!pip uninstall -y docx

In [ ]:
!pip install -q python-docx xlsxwriter

In [ ]:
import pandas as pd
import numpy as np
import pickle as pkl
import os

import docx              # python-docx — pull cluster names from manuscript
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

In [ ]:
# ---- Input paths ----
PATIENT_DEMOGRAPHICS_CSV = '/content/patient_demographics.csv'
PATIENT_COMORBIDITIES_CSV = '/content/patient_comorbidities.csv'
BASELINE_A1C_CSV          = '/content/baseline_a1c_all.csv'
BASELINE_BMI_CSV          = '/content/baseline_bmi_all.csv'

MONO_PKL    = '/content/monotherapy_patients.pkl'
DUAL_PKL    = '/content/dual_therapy_patients.pkl'
COMPLEX_PKL = '/content/complex_therapy_patients.pkl'
VARIANT_PKL = '/content/variant_therapy_patients.pkl'
GLP1_PKL    = '/content/GLP_1_therapy_patients.pkl'

MANUSCRIPT_DOCX = '/content/DOM T2D Manuscript v2026-04-16.docx'

# ---- Output paths ----
OUT_XLSX        = '/content/Cluster_Table1_Breakdown.xlsx'
OUT_HEATMAP_PNG = '/content/cluster_table1_heatmap.png'

# python-docx is 0-indexed: Table 1 = index 0, Table 2 (Race) = 1, Table 3 (Age) = 2.
DOCX_RACE_TABLE_INDEX = 1

In [ ]:
patient_demographics  = pd.read_csv(PATIENT_DEMOGRAPHICS_CSV)
patient_comorbidities = pd.read_csv(PATIENT_COMORBIDITIES_CSV)
baseline_a1c          = pd.read_csv(BASELINE_A1C_CSV)
baseline_bmi          = pd.read_csv(BASELINE_BMI_CSV)

with open(MONO_PKL,    'rb') as f: monotherapy_patients    = pkl.load(f)
with open(DUAL_PKL,    'rb') as f: dual_therapy_patients   = pkl.load(f)
with open(COMPLEX_PKL, 'rb') as f: complex_therapy_patients = pkl.load(f)
with open(VARIANT_PKL, 'rb') as f: variant_therapy_patients = pkl.load(f)
with open(GLP1_PKL,    'rb') as f: GLP_1_therapy_patients   = pkl.load(f)

# Parse dates
patient_comorbidities['date'] = pd.to_datetime(patient_comorbidities['date'], errors='coerce')

print('Loaded:')
print(f'  patient_demographics:  {len(patient_demographics):,} rows')
print(f'  patient_comorbidities: {len(patient_comorbidities):,} rows')
print(f'  baseline_a1c:          {len(baseline_a1c):,} rows')
print(f'  baseline_bmi:          {len(baseline_bmi):,} rows')
print(f'  Monotherapy clusters:  {len(monotherapy_patients)}')
print(f'  Dual Therapy clusters: {len(dual_therapy_patients)}')
print(f'  Complex Therapy clusters: {len(complex_therapy_patients)}')
print(f'  Variant Therapy clusters: {len(variant_therapy_patients)}')
print(f'  GLP-1 Therapy clusters:   {len(GLP_1_therapy_patients)}')

## Section 2 — Build patient → cluster assignment table

We flatten each treatment-group dictionary into a long-form table with one row
per patient and columns `Group`, `group_rank`, `patient_id`. The five group
dictionaries cover the 30 primary clusters (N = 9,327). This is exactly the
patient set summarized in the revised Table 1.

In [ ]:
GROUP_ORDER = ['Monotherapy', 'Dual Therapy', 'Complex Therapy',
               'Variant Therapy', 'GLP-1 Therapy']

GROUP_DICTS = {
    'Monotherapy':     monotherapy_patients,
    'Dual Therapy':    dual_therapy_patients,
    'Complex Therapy': complex_therapy_patients,
    'Variant Therapy': variant_therapy_patients,
    'GLP-1 Therapy':   GLP_1_therapy_patients,
}

def prep_group_df(group_dict, group_name):
    """Convert {cluster_id: [patient_ids]} to a long-form DataFrame.

    `group_rank` matches the position used in manuscript Tables 2/3 and eTable 2
    (1-indexed within group, ordered by the dict's insertion order).
    """
    rows = []
    for rank, (_, pid_list) in enumerate(group_dict.items(), start=1):
        for pid in pid_list:
            rows.append({'Group': group_name, 'group_rank': rank, 'patient_id': str(pid)})
    return pd.DataFrame(rows)

cluster_long = pd.concat(
    [prep_group_df(GROUP_DICTS[g], g) for g in GROUP_ORDER],
    ignore_index=True,
)

N_PRIMARY = cluster_long['patient_id'].nunique()
print(f'Primary cohort across 30 clusters: {N_PRIMARY:,} unique patients.')
assert N_PRIMARY == 9327, (
    f'Expected 9,327 patients per manuscript Table 1; got {N_PRIMARY}. '
    f'Check that the cluster pkls correspond to the current cohort.'
)
print(f'Total cluster assignments: {len(cluster_long):,} (should equal {N_PRIMARY:,})')

## Section 3 — Pull cluster names from the manuscript

Mirrors the approach used in `SLR_Tables_with_FDR.ipynb`: extract cluster names
directly from manuscript Table 2 (`python-docx` table index 1) so row labels
stay in sync with the published version.

In [ ]:
CLUSTER_PREFIX_TO_GROUP = {
    'Mono:':    'Monotherapy',
    'Dual:':    'Dual Therapy',
    'Complex:': 'Complex Therapy',
    'Variant:': 'Variant Therapy',
    'GLP-1:':   'GLP-1 Therapy',
}

def _infer_group(cluster_name):
    for prefix, group in CLUSTER_PREFIX_TO_GROUP.items():
        if cluster_name.startswith(prefix):
            return group
    raise ValueError(f'Unknown cluster prefix: {cluster_name!r}')

doc = docx.Document(MANUSCRIPT_DOCX)
race_table = doc.tables[DOCX_RACE_TABLE_INDEX]

cluster_name_rows = []
for row in race_table.rows[1:]:  # row 0 is the column header
    cells_in_row = [c.text.strip() for c in row.cells]
    cluster_name = cells_in_row[0]
    group_rank   = int(cells_in_row[1])
    group        = _infer_group(cluster_name)
    cluster_name_rows.append({
        'Group':        group,
        'group_rank':   group_rank,
        'cluster_name': cluster_name,
    })

cluster_names_df = pd.DataFrame(cluster_name_rows)
assert len(cluster_names_df) == 30, f'Expected 30 cluster names; got {len(cluster_names_df)}'
print(f'Pulled {len(cluster_names_df)} cluster names from {MANUSCRIPT_DOCX}.')
cluster_names_df.head(6)

## Section 4 — Build patient-level analytic table

Join cluster assignments to demographics, baseline HbA1c, baseline BMI, and
pre-June-2019 comorbidity flags. This uses the same data-source and definition
conventions as `Table1_Construction.ipynb` (`patient_id` cast to string for
consistent joining; comorbidities aggregated to the patient level with the
documented before-June-1-2019 window).

In [ ]:
WINDOW_END = pd.Timestamp('2019-06-01')

# ---- Demographics: compute age at 2019, normalize patient_id ----
dem = patient_demographics.copy()
dem['patient_id'] = dem['patient_id'].astype(str)
dem['age_2019'] = 2019 - pd.to_numeric(dem['year_of_birth'], errors='coerce')
dem = dem[['patient_id', 'sex', 'race/ethnicity',
           'patient_regional_location', 'age_2019']]

# ---- Baseline HbA1c and BMI ----
a1c = baseline_a1c.copy()
a1c['patient_id'] = a1c['patient_id'].astype(str)
a1c = a1c[['patient_id', 'baseline_a1c_value']]

bmi = baseline_bmi.copy()
bmi['patient_id'] = bmi['patient_id'].astype(str)
bmi = bmi[['patient_id', 'baseline_bmi_value']]

# ---- Comorbidities: HF/CKD documented on or before June 1, 2019 ----
# Mirrors Table1_Construction.ipynb Section 4, Block 3.
com = patient_comorbidities.copy()
com['patient_id'] = com['patient_id'].astype(str)
com_window = com[com['date'] <= WINDOW_END].copy()
for col in ['HF', 'CKD']:
    com_window[col] = com_window[col].astype(str).str.strip().str.upper().isin(
        ['TRUE', '1', 'T', 'Y', 'YES'])

hf_ids  = set(com_window.loc[com_window['HF'],  'patient_id'])
ckd_ids = set(com_window.loc[com_window['CKD'], 'patient_id'])

# ---- Assemble patient-level table ----
pt = (cluster_long
      .merge(dem, on='patient_id', how='left')
      .merge(a1c, on='patient_id', how='left')
      .merge(bmi, on='patient_id', how='left'))
pt['has_HF']  = pt['patient_id'].isin(hf_ids)
pt['has_CKD'] = pt['patient_id'].isin(ckd_ids)

print(f'Patient-level table: {len(pt):,} rows.')
print(f'  Missing demographics:    {pt["sex"].isna().sum()}')
print(f'  Missing baseline_a1c:    {pt["baseline_a1c_value"].isna().sum()}')
print(f'  Missing baseline_bmi:    {pt["baseline_bmi_value"].isna().sum()}')
print(f'  Patients with HF flag:   {pt["has_HF"].sum()}')
print(f'  Patients with CKD flag:  {pt["has_CKD"].sum()}')

## Section 5 — Compute per-cluster summary statistics

For each `(Group, group_rank)`, compute the variables shown in manuscript Table
1. Race/ethnicity labels and region levels follow the same convention as
`Table1_Construction.ipynb` Section 5.

In [ ]:
# Race/ethnicity (matches Table1_Construction.ipynb labels)
RACE_INPUT_LEVELS = ['White', 'Hispanic', 'Asian', 'Black', 'Other']
RACE_DISPLAY = {
    'Black':    '% African American/Black, NH',
    'Asian':    '% Asian, NH',
    'Hispanic': '% Hispanic/Latinx',
    'White':    '% White, NH',
    'Other':    '% Other, NH',
}
REGION_LEVELS = ['Midwest', 'South', 'West', 'Northeast']
REGION_DISPLAY = {r: f'% {r}' for r in REGION_LEVELS}

def _mean_sd(series):
    s = pd.to_numeric(series, errors='coerce').dropna()
    if len(s) == 0:
        return (np.nan, np.nan)
    return (float(s.mean()), float(s.std()))

def _pct_female(series):
    s = series.astype(str).str.upper().str.strip()
    return 100.0 * s.isin(['F', 'FEMALE']).mean() if len(s) else np.nan

def _pct_match(series, value):
    s = series.astype(str).str.strip()
    return 100.0 * (s == value).mean() if len(s) else np.nan

def _pct_bool(series):
    s = series.fillna(False).astype(bool)
    return 100.0 * s.mean() if len(s) else np.nan

records = []
for (group, group_rank), sub in pt.groupby(['Group', 'group_rank']):
    n = len(sub)
    age_m, age_sd = _mean_sd(sub['age_2019'])
    a1c_m, a1c_sd = _mean_sd(sub['baseline_a1c_value'])
    bmi_m, bmi_sd = _mean_sd(sub['baseline_bmi_value'])
    rec = {
        'Group':            group,
        'group_rank':       group_rank,
        'n':                n,
        '% Female':         _pct_female(sub['sex']),
        'Age, mean (SD)':   f'{age_m:.1f} ({age_sd:.1f})' if not np.isnan(age_m) else '',
        '_age_mean':        age_m,
        'HbA1c, mean (SD)': f'{a1c_m:.2f} ({a1c_sd:.2f})' if not np.isnan(a1c_m) else '',
        '_a1c_mean':        a1c_m,
        'BMI, mean (SD)':   f'{bmi_m:.1f} ({bmi_sd:.1f})' if not np.isnan(bmi_m) else '',
        '_bmi_mean':        bmi_m,
        '% HF':             _pct_bool(sub['has_HF']),
        '% CKD':            _pct_bool(sub['has_CKD']),
    }
    for race in RACE_INPUT_LEVELS:
        rec[RACE_DISPLAY[race]] = _pct_match(sub['race/ethnicity'], race)
    for region in REGION_LEVELS:
        rec[REGION_DISPLAY[region]] = _pct_match(sub['patient_regional_location'], region)
    records.append(rec)

stats_df = pd.DataFrame(records)
stats_df['Group'] = pd.Categorical(stats_df['Group'], categories=GROUP_ORDER, ordered=True)
stats_df = stats_df.sort_values(['Group', 'group_rank']).reset_index(drop=True)
stats_df = stats_df.merge(cluster_names_df, on=['Group', 'group_rank'], how='left')
assert stats_df['cluster_name'].notna().all()

print(f'Stats computed for {len(stats_df)} clusters.')
stats_df.head(3)

## Section 6 (Experiment A) — Long-form eTable

A 30-row table with one column per Table 1 variable, plus a final cohort-overall
row that should match Table 1 in the manuscript (sanity check: the cohort-row
values can be compared cell-by-cell against Table 1).

Columns:

- **Cluster identifier:** Cluster name, group rank (#), N.
- **Demographics:** Mean age (SD), % female, race/ethnicity breakdown (5 levels), region breakdown (4 levels).
- **Clinical:** Mean baseline HbA1c (SD), mean baseline BMI (SD).
- **Comorbidities:** % HF, % CKD.

In [ ]:
# ---- Cohort-overall reference row (should match Table 1) ----
age_m_all, age_sd_all = _mean_sd(pt['age_2019'])
a1c_m_all, a1c_sd_all = _mean_sd(pt['baseline_a1c_value'])
bmi_m_all, bmi_sd_all = _mean_sd(pt['baseline_bmi_value'])

cohort_row = {
    'Cluster':          'Cohort overall',
    '#':                '',
    'n':                len(pt),
    'Age, mean (SD)':   f'{age_m_all:.1f} ({age_sd_all:.1f})',
    '% Female':         _pct_female(pt['sex']),
    'HbA1c, mean (SD)': f'{a1c_m_all:.2f} ({a1c_sd_all:.2f})',
    'BMI, mean (SD)':   f'{bmi_m_all:.1f} ({bmi_sd_all:.1f})',
    '% HF':             _pct_bool(pt['has_HF']),
    '% CKD':            _pct_bool(pt['has_CKD']),
}
for race in RACE_INPUT_LEVELS:
    cohort_row[RACE_DISPLAY[race]] = _pct_match(pt['race/ethnicity'], race)
for region in REGION_LEVELS:
    cohort_row[REGION_DISPLAY[region]] = _pct_match(pt['patient_regional_location'], region)

# ---- Column order ----
display_cols = [
    'Cluster', '#', 'n',
    'Age, mean (SD)', '% Female',
    '% African American/Black, NH', '% Asian, NH', '% Hispanic/Latinx',
    '% White, NH', '% Other, NH',
    '% Midwest', '% South', '% West', '% Northeast',
    'HbA1c, mean (SD)', 'BMI, mean (SD)',
    '% HF', '% CKD',
]

display_rows = []
for _, r in stats_df.iterrows():
    row = {
        'Cluster':          r['cluster_name'],
        '#':                int(r['group_rank']),
        'n':                int(r['n']),
        'Age, mean (SD)':   r['Age, mean (SD)'],
        '% Female':         r['% Female'],
        'HbA1c, mean (SD)': r['HbA1c, mean (SD)'],
        'BMI, mean (SD)':   r['BMI, mean (SD)'],
        '% HF':             r['% HF'],
        '% CKD':            r['% CKD'],
    }
    for race in RACE_INPUT_LEVELS:
        row[RACE_DISPLAY[race]] = r[RACE_DISPLAY[race]]
    for region in REGION_LEVELS:
        row[REGION_DISPLAY[region]] = r[REGION_DISPLAY[region]]
    display_rows.append(row)

display_rows.append(cohort_row)
exp_A_df = pd.DataFrame(display_rows, columns=display_cols)

# Format percentages to one decimal.
pct_cols = ['% Female',
            '% African American/Black, NH', '% Asian, NH', '% Hispanic/Latinx',
            '% White, NH', '% Other, NH',
            '% Midwest', '% South', '% West', '% Northeast',
            '% HF', '% CKD']
for c in pct_cols:
    exp_A_df[c] = exp_A_df[c].apply(lambda v: f'{v:.1f}' if isinstance(v, (int, float)) and pd.notna(v) else (v if v != '' else ''))

exp_A_df

## Section 7 (Experiment B) — Standardized heatmap

A heatmap with 30 clusters as rows and the key characteristics as columns. Cell
color encodes the cluster's value standardized as a z-score across clusters (red
= above cohort norm, blue = below); cell text shows the raw value. Therapy-group
blocks are visually separated with horizontal lines, in the same order as
Experiment A and the manuscript tables.

Region is omitted here for visual clarity (it would add four columns of
moderate-variance data without bearing on the central disparity narrative). It
remains available in Experiment A.

In [ ]:
heatmap_cols = [
    ('Age (yr)',        '_age_mean',                       '.1f'),
    ('% Female',        '% Female',                        '.1f'),
    ('% White',         RACE_DISPLAY['White'],             '.1f'),
    ('% Black',         RACE_DISPLAY['Black'],             '.1f'),
    ('% Hispanic',      RACE_DISPLAY['Hispanic'],          '.1f'),
    ('% Asian',         RACE_DISPLAY['Asian'],             '.1f'),
    ('% Other',         RACE_DISPLAY['Other'],             '.1f'),
    ('HbA1c (%)',       '_a1c_mean',                       '.2f'),
    ('BMI (kg/m²)', '_bmi_mean',                      '.1f'),
    ('% HF',            '% HF',                            '.1f'),
    ('% CKD',           '% CKD',                           '.1f'),
]

row_labels = [f"{r['cluster_name']} (#{int(r['group_rank'])})" for _, r in stats_df.iterrows()]
col_labels = [c[0] for c in heatmap_cols]
raw_mat    = np.array([[stats_df.loc[i, c[1]] for c in heatmap_cols]
                       for i in stats_df.index], dtype=float)

# z-score per column across the 30 clusters.
col_means = np.nanmean(raw_mat, axis=0)
col_stds  = np.nanstd(raw_mat, axis=0, ddof=0)
col_stds[col_stds == 0] = 1.0
z_mat = (raw_mat - col_means) / col_stds

fig_height = max(10, 0.30 * len(row_labels) + 2)
fig_width  = 1.1 + 0.95 * len(col_labels)
fig, ax = plt.subplots(figsize=(fig_width, fig_height))
vmax = float(np.nanmax(np.abs(z_mat)))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
im = ax.imshow(z_mat, cmap='RdBu_r', norm=norm, aspect='auto')

# Annotate with raw values.
for i in range(raw_mat.shape[0]):
    for j, (_, _, fmt) in enumerate(heatmap_cols):
        v = raw_mat[i, j]
        txt = '' if np.isnan(v) else format(v, fmt)
        z = z_mat[i, j]
        color = 'white' if (not np.isnan(z) and abs(z) > 1.2) else 'black'
        ax.text(j, i, txt, ha='center', va='center', fontsize=8, color=color)

ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, rotation=40, ha='right', fontsize=9)
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=8)

# Therapy-group separators.
group_seq = stats_df['Group'].tolist()
for i in range(1, len(group_seq)):
    if group_seq[i] != group_seq[i-1]:
        ax.axhline(i - 0.5, color='black', linewidth=1.2)

cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label('Standardized score (per-column z-score across 30 clusters)', fontsize=9)
ax.set_title('Cluster-Level Patient Characteristics — Standardized View',
             fontsize=12, pad=12)
plt.tight_layout()
plt.savefig(OUT_HEATMAP_PNG, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved heatmap to {OUT_HEATMAP_PNG}.')

## Section 8 — Export to Excel

Two sheets:

- `Experiment A — Long Table` — the printable eTable.
- `Experiment B — Heatmap Data` — the underlying raw and z-scored matrices, so
  the figure can be reproduced or re-styled later.

In [ ]:
heatmap_raw_df = pd.DataFrame(raw_mat,
                              columns=[c[0] for c in heatmap_cols],
                              index=row_labels)
heatmap_z_df   = pd.DataFrame(z_mat,
                              columns=[f'z: {c[0]}' for c in heatmap_cols],
                              index=row_labels)
heatmap_export = pd.concat([heatmap_raw_df, heatmap_z_df], axis=1)
heatmap_export.index.name = 'Cluster'
heatmap_export = heatmap_export.reset_index()

with pd.ExcelWriter(OUT_XLSX, engine='xlsxwriter') as writer:
    wb = writer.book
    title_fmt  = wb.add_format({'bold': True, 'font_size': 13, 'align': 'left'})
    header_fmt = wb.add_format({'bold': True, 'font_size': 10, 'bottom': 1})
    body_fmt   = wb.add_format({'font_size': 10})

    # ---- Sheet 1: Experiment A ----
    sheet_a = 'Experiment A - Long Table'
    exp_A_df.to_excel(writer, sheet_name=sheet_a, index=False, startrow=2)
    ws = writer.sheets[sheet_a]
    ws.merge_range(0, 0, 0, len(exp_A_df.columns) - 1,
                   'Cluster-Level Patient Characteristics (Long Table)', title_fmt)
    ws.set_column(0, 0, 36, body_fmt)
    ws.set_column(1, 1,  4, body_fmt)
    ws.set_column(2, 2,  7, body_fmt)
    ws.set_column(3, len(exp_A_df.columns) - 1, 16, body_fmt)
    for col_idx, col_name in enumerate(exp_A_df.columns):
        ws.write(2, col_idx, col_name, header_fmt)

    # ---- Sheet 2: Experiment B (data) ----
    sheet_b = 'Experiment B - Heatmap Data'
    heatmap_export.to_excel(writer, sheet_name=sheet_b, index=False, startrow=2)
    ws = writer.sheets[sheet_b]
    ws.merge_range(0, 0, 0, len(heatmap_export.columns) - 1,
                   'Cluster-Level Heatmap — Raw and Z-Scored Values', title_fmt)
    ws.set_column(0, 0, 42, body_fmt)
    ws.set_column(1, len(heatmap_export.columns) - 1, 12, body_fmt)
    for col_idx, col_name in enumerate(heatmap_export.columns):
        ws.write(2, col_idx, col_name, header_fmt)

print(f'Exported to {OUT_XLSX}.')
print(f'Heatmap PNG: {OUT_HEATMAP_PNG}.')

## Section 9 — Notes for response letter and supplement placement

- **Experiment A** would fit naturally as a new eTable adjacent to eTable 2
  (top prescriptions / HF / CKD per cluster). Adding it as eTable 11 (next
  available number after eTable 10) is the simplest insertion.
- **Experiment B** would fit as a new eFigure — the reviewer explicitly asked
  for a graphical presentation. It pairs well with the existing per-cluster
  three-panel figures.
- The two are not mutually exclusive — Experiment A is more comprehensive,
  Experiment B is more visually scannable. If page budget allows, include the
  heatmap in the supplement and reference the long table as a downloadable file.
- **Sanity check.** The `Cohort overall` row of Experiment A should reproduce
  the values in manuscript Table 1 exactly (same data sources, same baseline
  window, same comorbidity definition). If it doesn't, check that the input
  CSVs and pkls are from the current cohort version.